# CosyVoice 2 — Indian English TTS (Zero-Shot + Fine-Tune)

**Apache 2.0 license** — code AND weights. Enterprise/commercial use confirmed.

Works on **both Colab and RunPod**.

| Step | What | Time | Decision Point |
|------|------|------|-----------|
| 1 | Setup + HF login + data download | ~15 min | Data ready? |
| 2 | Install + load CosyVoice 2 | ~10 min | Model loads? |
| 3 | Pick reference voices | ~5 min | Listen and choose |
| 4 | Generate podcast (NO training) | ~10 min | **Good enough for pilot?** |
| 5 | Fine-tune (optional) | ~1-2 hrs | Better than zero-shot? |

## Step 1: GPU + HuggingFace Login + Google Drive + Data Download

In [ ]:
import os, sys, shutil, glob
import torch

# --- Detect environment ---
IS_COLAB = 'google.colab' in sys.modules or os.path.exists('/content')
BASE = '/content' if IS_COLAB else '/workspace'
print(f"Environment: {'Colab' if IS_COLAB else 'RunPod/Other'}")
print(f"Base dir: {BASE}")

# --- GPU Check ---
assert torch.cuda.is_available(), "No GPU detected!"
print(f"GPU: {torch.cuda.get_device_name(0)}")

# --- HuggingFace Login ---
try:
    if IS_COLAB:
        from google.colab import userdata
        os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
except Exception:
    pass

if not os.environ.get('HF_TOKEN'):
    print("HF_TOKEN not set. Trying interactive login...")
    from huggingface_hub import login
    login()
else:
    from huggingface_hub import HfApi
    try:
        print(f"HF user: {HfApi().whoami()['name']}")
    except Exception:
        from huggingface_hub import login
        login()

# --- Backup directory ---
if IS_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    BACKUP_DIR = '/content/drive/MyDrive/indian_tts_cosyvoice2'
else:
    BACKUP_DIR = f'{BASE}/backup_cosyvoice2'
os.makedirs(BACKUP_DIR, exist_ok=True)
print(f"Backup dir: {BACKUP_DIR}")

# --- Download Svarah Indian English Data ---
os.chdir(BASE)
REPO_DIR = f'{BASE}/indian_tts'
DATA_DIR = f'{BASE}/data'

if not os.path.exists(REPO_DIR):
    !git clone https://github.com/seetha0712/text2speech_1.git {REPO_DIR}
os.chdir(REPO_DIR)
!git checkout claude/custom-indian-tts-model-TUAjJ -q
!git pull origin claude/custom-indian-tts-model-TUAjJ -q
!pip install -q num2words soundfile
!pip install -q -e . 2>&1 | tail -1
!apt-get install -qq espeak-ng > /dev/null 2>&1

if not os.path.exists(f'{DATA_DIR}/train.txt'):
    print("\nDownloading Svarah Indian English data...")
    !python -m indian_tts.data.preprocess --source svarah --output {DATA_DIR}
else:
    with open(f'{DATA_DIR}/train.txt') as f:
        n = sum(1 for l in f if l.strip() and not l.startswith('#'))
    print(f"\nSvarah data already downloaded: {n} training samples")

print("\nStep 1 complete!")

In [ ]:
# System dependencies
!apt-get -qq install -y sox libsox-dev ffmpeg > /dev/null 2>&1

# Clone CosyVoice with submodules
COSYVOICE_DIR = f'{BASE}/CosyVoice'
os.chdir(BASE)
if not os.path.exists(f'{COSYVOICE_DIR}/.git'):
    !rm -rf {COSYVOICE_DIR}
    !git clone --recursive https://github.com/FunAudioLLM/CosyVoice.git {COSYVOICE_DIR}
    !cd {COSYVOICE_DIR} && git submodule update --init --recursive

# Install dependencies (skip torch reinstall)
!cd {COSYVOICE_DIR} && pip install -q -r requirements.txt \
    --ignore-installed torch torchaudio 2>&1 | tail -3

print("\nCosyVoice install complete!")

In [ ]:
# Set up Python path and download model
import sys
sys.path.insert(0, COSYVOICE_DIR)
sys.path.insert(0, f'{COSYVOICE_DIR}/third_party/Matcha-TTS')
os.chdir(COSYVOICE_DIR)

from huggingface_hub import snapshot_download
MODEL_DIR = f'{COSYVOICE_DIR}/pretrained_models/CosyVoice2-0.5B'
snapshot_download('FunAudioLLM/CosyVoice2-0.5B', local_dir=MODEL_DIR)
print("Model downloaded!")

In [ ]:
# Load model
os.chdir(COSYVOICE_DIR)
from cosyvoice.cli.cosyvoice import CosyVoice2
import torchaudio

cosyvoice = CosyVoice2('pretrained_models/CosyVoice2-0.5B')
print(f"Model loaded! Sample rate: {cosyvoice.sample_rate}")
print("CHECKPOINT: Model loads successfully.")

## Step 2: Select Reference Voices from Svarah

We'll pick one clean male and one clean female clip from your downloaded Svarah data as reference voices.

In [ ]:
import glob
import soundfile as sf
import numpy as np
import IPython.display as ipd

# Find reference clips from Svarah data
male_wavs = sorted(glob.glob(f'{DATA_DIR}/svarah/male/svarah_*.wav'))
female_wavs = sorted(glob.glob(f'{DATA_DIR}/svarah/female/svarah_*.wav'))

print(f"Male clips available: {len(male_wavs)}")
print(f"Female clips available: {len(female_wavs)}")

if not male_wavs or not female_wavs:
    print("ERROR: No Svarah clips found! Re-run Step 1.")
else:
    # Pick clips that are 5-12 seconds (ideal for reference)
    def find_good_reference(wav_list, min_dur=5, max_dur=12):
        for path in wav_list:
            data, sr = sf.read(path)
            dur = len(data) / sr
            if min_dur <= dur <= max_dur:
                return path, dur
        return wav_list[0], len(sf.read(wav_list[0])[0]) / sf.read(wav_list[0])[1]

    male_ref, male_dur = find_good_reference(male_wavs)
    female_ref, female_dur = find_good_reference(female_wavs)

    # Get transcripts from manifest
    male_ref_text = ""
    female_ref_text = ""
    for manifest in [f'{DATA_DIR}/train.txt', f'{DATA_DIR}/val.txt', f'{DATA_DIR}/test.txt']:
        if os.path.exists(manifest):
            with open(manifest) as f:
                for line in f:
                    line = line.strip()
                    if not line or line.startswith('#'):
                        continue
                    parts = line.split('|')
                    if parts[0] == male_ref:
                        male_ref_text = parts[2]
                    if parts[0] == female_ref:
                        female_ref_text = parts[2]

    print(f"\nMale ref: {male_ref} ({male_dur:.1f}s)")
    print(f"  Text: {male_ref_text[:80]}...")
    print(f"Female ref: {female_ref} ({female_dur:.1f}s)")
    print(f"  Text: {female_ref_text[:80]}...")

    print("\n[MALE reference]")
    ipd.display(ipd.Audio(male_ref))
    print("\n[FEMALE reference]")
    ipd.display(ipd.Audio(female_ref))

## Step 3: Generate Podcast — Zero-Shot (NO training!)

This uses the reference clips above to clone the voices. No fine-tuning needed.

In [ ]:
import time
import numpy as np
import soundfile as sf

PODCAST_SCRIPT = [
    ("female", "Welcome to AI India, the podcast where we explore how artificial intelligence is transforming our country. I am Priya."),
    ("male", "And I am Arjun. Today we are talking about something really exciting. The rise of Indian AI startups."),
    ("female", "That is right, Arjun. India now has over three hundred AI startups, and that number is growing every single month."),
    ("male", "What I find really interesting is that many of these companies are solving uniquely Indian problems. Like agriculture, healthcare in rural areas, and education."),
    ("female", "Absolutely. Take for example an AI system that can detect crop diseases just by looking at a photo taken on a farmer's mobile phone."),
    ("male", "And in healthcare, AI models are now screening for conditions like diabetic retinopathy and tuberculosis in areas where there are very few doctors available."),
    ("female", "The language barrier is another big challenge that AI is helping with. India has twenty two official languages and hundreds of dialects."),
    ("male", "Exactly. And that is precisely why building speech technology like text to speech systems in Indian languages is so important."),
    ("female", "Speaking of which, the progress in Indian language AI has been remarkable. Models can now understand and generate speech in Hindi, Tamil, Bengali, and many more."),
    ("male", "The government has also been supportive with initiatives to build open source datasets for Indian languages. This is a game changer."),
    ("female", "So what do you think is next for AI in India, Arjun?"),
    ("male", "I believe we will see AI becoming a part of everyday life. From voice assistants that truly understand Indian accents, to AI tutors that teach children in their mother tongue."),
    ("female", "That is a beautiful vision. And it all starts with building the right foundation, the right data, the right models, and the right talent."),
    ("male", "Could not agree more. India has the talent, and now we are building the tools."),
    ("female", "That is all for today's episode of AI India. Thank you for listening, and we will see you next week."),
    ("male", "Goodbye everyone, and keep innovating!"),
]

OUTPUT_DIR = f'{BASE}/outputs/cosyvoice2_podcast'
os.makedirs(OUTPUT_DIR, exist_ok=True)
sr = cosyvoice.sample_rate

all_segments = []
silence_between = np.zeros(int(sr * 0.6))

print("Generating podcast with CosyVoice 2 (zero-shot)...\n")
start = time.time()

for i, (speaker, text) in enumerate(PODCAST_SCRIPT):
    name = "Priya" if speaker == "female" else "Arjun"
    ref_wav = female_ref if speaker == "female" else male_ref
    ref_txt = female_ref_text if speaker == "female" else male_ref_text

    gen_start = time.time()
    audio_chunks = []
    for result in cosyvoice.inference_zero_shot(text, ref_txt, ref_wav, stream=False):
        audio_chunks.append(result['tts_speech'].squeeze().numpy())

    audio = np.concatenate(audio_chunks) if audio_chunks else np.zeros(sr)
    gen_time = time.time() - gen_start
    duration = len(audio) / sr

    print(f"  [{name:5s}] {duration:.1f}s (gen: {gen_time:.1f}s) | {text[:50]}...")
    sf.write(f'{OUTPUT_DIR}/line_{i:02d}_{speaker}.wav', audio, sr)

    if i > 0:
        all_segments.append(silence_between)
    all_segments.append(audio)

full_audio = np.concatenate(all_segments)
PODCAST_PATH = f'{OUTPUT_DIR}/podcast_full.wav'
sf.write(PODCAST_PATH, full_audio, sr)

total_time = time.time() - start
total_dur = len(full_audio) / sr
print(f"\nPodcast generated!")
print(f"  Duration: {total_dur:.0f}s ({total_dur/60:.1f} min)")
print(f"  Generation time: {total_time:.0f}s")
print(f"  RTF: {total_time/total_dur:.2f}x")

# Auto-backup
shutil.copy2(PODCAST_PATH, os.path.join(BACKUP_DIR, 'podcast_zero_shot.wav'))
print(f"  Backed up to: {BACKUP_DIR}")

In [ ]:
# LISTEN TO THE PODCAST
print("=" * 60)
print("  CosyVoice 2 - Zero-Shot Podcast (NO training)")
print("  Arjun (male) & Priya (female) discuss AI in India")
print("=" * 60)

print("\nFull podcast:")
ipd.display(ipd.Audio(PODCAST_PATH))

print("\nIndividual lines:")
for i in range(min(4, len(PODCAST_SCRIPT))):
    speaker, text = PODCAST_SCRIPT[i]
    name = "Priya" if speaker == "female" else "Arjun"
    wav_path = f'{OUTPUT_DIR}/line_{i:02d}_{speaker}.wav'
    print(f"\n  [{name}] {text[:60]}...")
    ipd.display(ipd.Audio(wav_path))

### DECISION POINT

**Listen above.** Is the quality good enough for your pilot?

- **YES** -> Skip to the "Save Model" section below. You're done!
- **Accent not Indian enough** -> Try different reference clips (re-run Step 2 with different indices)
- **Needs more consistency** -> Proceed to Step 4 (Fine-tuning)

---
## Step 4: Fine-Tune (Optional — only if zero-shot isn't good enough)

This prepares the Svarah data in CosyVoice format and fine-tunes the LLM component.

In [ ]:
# Prepare Svarah data in Kaldi format for CosyVoice fine-tuning
import os
from collections import defaultdict

ft_dir = '/content/cosyvoice_ft_data'
os.makedirs(ft_dir, exist_ok=True)

wav_scp = []
text_lines = []
utt2spk = []
spk2utt = defaultdict(list)

for manifest in ['/content/data/train.txt']:
    with open(manifest) as f:
        for idx, line in enumerate(f):
            line = line.strip()
            if not line or line.startswith('#'):
                continue
            parts = line.split('|')
            audio_path, speaker_id, text = parts[0], parts[1], parts[2]
            spk = 'male' if speaker_id == '0' else 'female'
            utt_id = f'utt_{idx:06d}'

            wav_scp.append(f'{utt_id} {audio_path}')
            text_lines.append(f'{utt_id} {text}')
            utt2spk.append(f'{utt_id} {spk}')
            spk2utt[spk].append(utt_id)

with open(f'{ft_dir}/wav.scp', 'w') as f:
    f.write('\n'.join(wav_scp))
with open(f'{ft_dir}/text', 'w') as f:
    f.write('\n'.join(text_lines))
with open(f'{ft_dir}/utt2spk', 'w') as f:
    f.write('\n'.join(utt2spk))
with open(f'{ft_dir}/spk2utt', 'w') as f:
    for spk, utts in spk2utt.items():
        f.write(f'{spk} {" ".join(utts)}\n')

print(f"Prepared {len(wav_scp)} utterances")
print(f"  Male: {len(spk2utt['male'])}")
print(f"  Female: {len(spk2utt['female'])}")
print(f"  Data dir: {ft_dir}")

In [ ]:
# Step 1: Extract speaker embeddings
os.chdir('/content/CosyVoice')
!python tools/extract_embedding.py --dir {ft_dir} \
    --onnx_path pretrained_models/CosyVoice2-0.5B/campplus.onnx

In [ ]:
# Step 2: Extract speech tokens
!python tools/extract_speech_token.py --dir {ft_dir} \
    --onnx_path pretrained_models/CosyVoice2-0.5B/speech_tokenizer_v2.onnx

In [ ]:
# Step 3: Convert to parquet
!python tools/make_parquet_list.py --num_utts_per_parquet 1000 \
    --num_processes 4 --src_dir {ft_dir} --des_dir {ft_dir}/parquet

In [ ]:
# Step 4: Fine-tune the LLM component
# This takes ~1-2 hours on A100
!torchrun --nnodes=1 --nproc_per_node=1 \
    --rdzv_id=1986 --rdzv_backend="c10d" --rdzv_endpoint="localhost:1234" \
    cosyvoice/bin/train.py \
    --train_engine torch_ddp \
    --config examples/libritts/cosyvoice2/conf/cosyvoice2.yaml \
    --train_data {ft_dir}/parquet/data.list \
    --cv_data {ft_dir}/parquet/data.list \
    --qwen_pretrain_path pretrained_models/CosyVoice2-0.5B/CosyVoice-BlankEN \
    --onnx_path pretrained_models/CosyVoice2-0.5B \
    --model llm \
    --checkpoint pretrained_models/CosyVoice2-0.5B/llm.pt \
    --model_dir /content/outputs/cosyvoice2_ft/llm \
    --tensorboard_dir /content/outputs/cosyvoice2_ft/tensorboard \
    --ddp.dist_backend nccl \
    --num_workers 2 --prefetch 100 --pin_memory --use_amp

---
## Save Model to Google Drive

In [ ]:
from google.colab import drive
import shutil
drive.mount('/content/drive')

backup_dir = '/content/drive/MyDrive/indian_tts_cosyvoice2'
os.makedirs(backup_dir, exist_ok=True)

# Save podcast
shutil.copy2('/content/outputs/cosyvoice2_podcast/podcast_full.wav', backup_dir)

# Save fine-tuned model (if it exists)
ft_ckpts = sorted(glob.glob('/content/outputs/cosyvoice2_ft/llm/*.pt'))
if ft_ckpts:
    shutil.copy2(ft_ckpts[-1], backup_dir)
    print(f"Fine-tuned checkpoint saved: {os.path.basename(ft_ckpts[-1])}")

print(f"Saved to: {backup_dir}")